In [0]:
%python

import pyspark.sql.functions as F
from pyspark.sql.types import * 

In [0]:
%python
# checkpoint dirctory for structured streaming

dbutils.fs.mkdirs("/Volumes/lakehouse/01_raw/raw/_checkpoint")

In [0]:
%python
source_path = '/Volumes/lakehouse/01_raw/raw/'
checkpoint_path = '/Volumes/lakehouse/01_raw/raw/_checkpoint'


In [0]:
%python
source_schema = StructType([
    StructField('bbox', ArrayType(DoubleType(), True), True),
    StructField('features', ArrayType(
        StructType([
            StructField('geometry', StructType([
                StructField('coordinates', ArrayType(DoubleType(), True), True),
                StructField('type', StringType(), True)
            ]), True),
            StructField('id', StringType(), True),
            StructField('properties', StructType([
                StructField('alert', StringType(), True),
                StructField('cdi', DoubleType(), True),
                StructField('code', StringType(), True),
                StructField('detail', StringType(), True),
                StructField('dmin', DoubleType(), True),
                StructField('felt', LongType(), True),
                StructField('gap', LongType(), True),
                StructField('ids', StringType(), True),
                StructField('mag', DoubleType(), True),
                StructField('magType', StringType(), True),
                StructField('mmi', DoubleType(), True),
                StructField('net', StringType(), True),
                StructField('nst', LongType(), True),
                StructField('place', StringType(), True),
                StructField('rms', DoubleType(), True),
                StructField('sig', LongType(), True),
                StructField('sources', StringType(), True),
                StructField('status', StringType(), True),
                StructField('time', LongType(), True),
                StructField('title', StringType(), True),
                StructField('tsunami', LongType(), True),
                StructField('type', StringType(), True),
                StructField('types', StringType(), True),
                StructField('tz', StringType(), True),
                StructField('updated', LongType(), True),
                StructField('url', StringType(), True)
            ]), True),
            StructField('type', StringType(), True)
        ]), True
    ), True),
    StructField('metadata', StructType([
        StructField('api', StringType(), True),
        StructField('count', LongType(), True),
        StructField('generated', LongType(), True),
        StructField('status', LongType(), True),
        StructField('title', StringType(), True),
        StructField('url', StringType(), True)
    ]), True),
    StructField('type', StringType(), True)
])

In [0]:
%python
df = (spark.readStream
      .format("json")
      .schema(source_schema)
      .load(source_path+"/*.json"))

In [0]:
%python
dbutils.fs.rm(checkpoint_path, recurse=True)

In [0]:
%python
query = (df.writeStream
         .option("checkpointLocation", checkpoint_path)
         .trigger(availableNow=True) # Process all new files then stop (Batch-like)
         .toTable('lakehouse.`02_bronze`.earthquakes'))

In [0]:
%python
df = spark.read.option('inferSchema', True).json(path='/Volumes/lakehouse/01_raw/raw/earthquake_data_2026-08-29-05H-26M-48S.json')

In [0]:
%python
df.schema